In [11]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"

import torch
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import ViTFeatureExtractor, ViTForImageClassification, Trainer, TrainingArguments

# 1. Load the feature extractor
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')

# 2. Define image transforms for train/test (augmentation for train)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std)
])

# 3. Load your dataset (assuming it's in ./data/train and ./data/test)
from torchvision.datasets import ImageFolder

train_dataset = ImageFolder('./data/train', transform=train_transform)
test_dataset = ImageFolder('./data/test', transform=test_transform)

# 4. Create DataLoaders (optional for Trainer but good for manual eval)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8)

# 5. Prepare class names
label2id = {cls: idx for idx, cls in enumerate(train_dataset.classes)}
id2label = {idx: cls for cls, idx in label2id.items()}

# 6. Load ViT model for binary classification
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification"
)

# Override loss to use binary cross-entropy (with logits)
model.classifier = nn.Sequential(
    nn.Linear(model.classifier.in_features, 1)  # binary output
)

# 7. Convert ImageFolder to Hugging Face DatasetDict
from datasets import Dataset, DatasetDict

def torch_dataset_to_hf(dataset):
    # Load original PIL images (skip transforms here)
    return Dataset.from_dict({
        "image": [Image.open(path).convert('RGB') for path, _ in dataset.samples],
        "label": [label for _, label in dataset.samples]
    })

dataset = DatasetDict({
    "train": torch_dataset_to_hf(train_dataset),
    "test": torch_dataset_to_hf(test_dataset)
})

def transform(example):
    # Convert tensor to PIL if needed
    if isinstance(example['image'], torch.Tensor):
        example['image'] = transforms.ToPILImage()(example['image'])
    example['pixel_values'] = feature_extractor(example['image'], return_tensors="pt")['pixel_values'][0]
    return example

dataset = dataset.map(transform, batched=False)

# 9. Set format for Trainer
dataset.set_format(type='torch', columns=['pixel_values', 'label'])

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = (torch.sigmoid(torch.tensor(logits)).numpy() > 0.5).astype(int)
    return {"accuracy": (preds.flatten() == labels).mean()}

training_args = TrainingArguments(
    output_dir="./vit-binary-tb",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.evaluate()


/Users/dinachat/miniforge3/envs/vit_env/lib/python3.10/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'Image' is not defined